# 🤖 Topic 3: Your First Chatbot

## What I'm going to show you

In Topics 1 and 2 we made individual API calls. Every call was standalone - no
reusable structure, no way to swap the product context, no way to configure the
assistant's behaviour without editing the raw messages list each time.

In this topic I'm going to show you how to go from ad-hoc API calls to a
properly designed chatbot function. By the end, you will have a
`build_barclays_chatbot()` function that returns a ready-to-use assistant
pre-loaded with the best available Barclays product context.

## Learning Objectives

By the end of this topic you will be able to:

1. Write an effective system prompt that constrains an assistant to a domain
2. Explain what makes a system prompt work (persona, constraints, context, format)
3. Build a chatbot factory using Python closures
4. Use the factory to create a Barclays Customer Service Assistant

## What component of the assistant am I building today?

The **Chat Interface** - the conversation layer of the Barclays Product Knowledge
Assistant. This is what sits between the customer's question and the API call.
Every topic from here uses the pattern you build today.

## ⚙️ Section 0: Environment Setup

I'll install the packages we need and connect to the OpenAI API.

In [ ]:
# openai: the OpenAI Python SDK - pinned to the same version as Topics 1 and 2
# sagemaker==2.232.1 is the last classic SDK release where get_execution_role lives at the top level
# boto3 is the AWS SDK - provides SageMaker session and S3 utilities
# numpy<2: pinned to avoid breaking changes in numpy 2.x (standard for all topics)
!pip install -q "openai==2.32.0" "sagemaker==2.232.1" "boto3" "numpy<2"

In [ ]:
import os
import getpass

from openai import OpenAI   # main synchronous client class

# Load the OpenAI API key
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key and press Enter: ")

# The single shared client for the whole notebook
client = OpenAI()   # reads OPENAI_API_KEY from os.environ

MODEL = "gpt-4o"  # course-wide model constant

# Product snippet used as fallback context when Topic 2 chunks are not available.
# Note: Topic 2 chunks contain personal loan FAQ text only. If you load Topic 2 chunks,
# credit card questions may fall through to the system prompt's general knowledge.
# This expanded snippet includes both loan and credit card details for Topic 3.
BARCLAYS_PRODUCT_SNIPPET = """
Barclays Personal Loan - Product Summary (illustrative)
- Loan amounts: 1,000 GBP to 35,000 GBP
- Representative APR: 6.5% for loans of 7,500 GBP to 15,000 GBP
- Repayment terms: 1 to 5 years
- No arrangement fee
- Early repayment allowed with 30-day interest charge
- Barclaycard Rewards credit card: 27.9% APR variable purchase rate
- Minimum monthly repayment: greater of 1% of balance plus interest, or GBP 25
- Late payment fee: GBP 12 if minimum payment not received by due date
"""

# GPT-4o pricing constants - used in the stretch cost-tracking exercise
GPT4O_INPUT_PRICE_PER_1K  = 0.0025   # dollars per 1K input tokens
GPT4O_OUTPUT_PRICE_PER_1K = 0.0100   # dollars per 1K output tokens

print("Setup complete.")

## 🏗️ What Are We Building Today?

Here is where we are in the Barclays Product Knowledge Assistant build:

```
customer_question
      |
      v
[Chat Interface]         <-- YOU ARE HERE
      |    \
      |     system_prompt (constrains behaviour)
      |     context (product knowledge)
      v
[GPT-4o via OpenAI API]
      |
      v
answer
```

Right now our Topic 1 code calls the API directly. There is no reusable structure.
Watch what happens when a developer skips the system prompt entirely.

**Before (Topic 1 style - ad hoc):**
```python
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": customer_question}]
)
```

**After (what we build today):**
```python
barclays_bot = build_barclays_chatbot()
answer = barclays_bot("What is the interest rate on the personal loan?")
```

One function call. Correct persona. Product context injected. Constraints enforced.
That is the goal for this topic.

## ✍️ Section 1: System Prompt Craft

### The problem: what happens without a system prompt?

Let me show you what a Barclays customer service chatbot looks like without a
system prompt. I'm going to call the API with just a user message and no
instructions to the model about who it is or what it should do.

In [ ]:
# NAIVE APPROACH: no system prompt at all.
# Watch what the model does when it has no instructions about persona, tone,
# or constraints. The output may be helpful or may be dangerously off-brand.

test_question = "Can you help me with my Barclays account balance?"

response_no_system = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        # No system message - the model fills in its own default persona
        {
            "role": "user",
            "content": test_question
        }
    ],
    max_tokens=200,
    temperature=0.2
)

print("Question:", test_question)
print()
print("Answer (no system prompt):")
print(response_no_system.choices[0].message.content)
print()
print("Problems with this response:")
print("  - Model may suggest visiting branches, using apps, calling numbers it invented")
print("  - No constraint that it should only discuss Barclays products")
print("  - No financial advice boundary - it might give specific investment advice")
print("  - Tone may be wrong for a UK banking context")
print("  - No escalation path for sensitive queries")

### The four-part system prompt anatomy

A well-designed system prompt for a banking assistant has four parts. Each part
does a specific job. Getting any one of them wrong degrades the whole response.

```mermaid
graph TD
    SP["System Prompt"]
    SP --> P["1. Persona Block<br/>Who the assistant is<br/>Tone: formal but warm"]
    SP --> C["2. Constraint Block<br/>What it must NOT do<br/>Scope + advice boundary"]
    SP --> X["3. Context Block<br/>Where to find answers<br/>Product info injected here"]
    SP --> F["4. Format Block<br/>How to structure answers<br/>Length, lists, escalation"]

    P -->|"Sets"| R1["On-brand voice"]
    C -->|"Prevents"| R2["Scope drift + FCA violations"]
    X -->|"Grounds"| R3["Factual product answers"]
    F -->|"Controls"| R4["Response shape"]
```

The four parts are:

**1. Persona block** - who the assistant is and what its tone should be.
Formal-but-warm is the right balance for UK banking: professional enough to carry
authority, warm enough to avoid feeling like a legal document.

**2. Constraint block** - what the assistant must NOT do. Explicit negatives are
as important as positive instructions. For financial services in the UK, two
constraints are non-negotiable:
- "You discuss only Barclays products and services" (scope constraint)
- "You provide product information only, not personalised financial advice" (FCA-aligned advice boundary)

**3. Context block** - where the assistant should look for information. In this
topic the context is injected as text in the user message. In Topics 6-7 it comes
from the RAG retrieval step.

**4. Format block** - how responses should be structured. For a customer service
assistant: plain prose, no markdown bullet lists (they look odd in chat UIs),
responses under 150 words for simple queries. Escalation path for complex ones.

In [ ]:
# FULL WORKING DEMO - a well-crafted Barclays customer service system prompt.
# I will walk through each of the four parts so you can see exactly what it does.

BARCLAYS_SYSTEM_PROMPT = """You are a Barclays Product Knowledge Assistant.
Your role is to help customers understand Barclays products and services.

Persona and tone:
You are formal but warm - professional, clear, and friendly without being chatty.
You address the customer directly and use plain English. No jargon unless you
explain it in the same sentence.

Constraints - what you must not do:
- Only discuss Barclays products and services. If asked about competitors or
  unrelated topics, politely redirect: "I can only help with Barclays products
  and services."
- Provide product information only. You do not give personalised financial advice.
  If a customer asks which product is best for their situation, say:
  "I can share product details to help you compare, but for personalised advice
  please speak to a Barclays adviser."
- Never invent figures, rates, or terms. If the product information does not
  contain an answer, say: "I don't have that information to hand - please contact
  Barclays directly or visit barclays.co.uk for the latest details."

Format:
Keep answers to 3 sentences or fewer for simple factual questions.
For multi-part questions, answer each part in a short paragraph.
Do not use bullet lists unless the question explicitly asks for a list.
"""

# Now use the system prompt in a real call
product_context = BARCLAYS_PRODUCT_SNIPPET   # we will make this dynamic in Concept 3

test_messages = [
    {
        "role": "system",
        "content": BARCLAYS_SYSTEM_PROMPT
    },
    {
        "role": "user",
        # Context injection: product data + question together in the user message.
        # This is the same pattern as Topic 1 - Topics 6-7 will replace the
        # hardcoded context with retrieved chunks from ChromaDB.
        "content": f"Product information:\n{product_context}\n\nCustomer question: {test_question}"
    }
]

response_with_system = client.chat.completions.create(
    model="gpt-4o",
    messages=test_messages,
    max_tokens=300,
    temperature=0.2   # low temperature = consistent, factual answers
)

print("Question:", test_question)
print()
print("Answer (with system prompt):")
print(response_with_system.choices[0].message.content)
print()
print("Notice the differences: on-brand tone, correct constraints, no invented data.")

### 🔬 Lab 1: Write Your Own System Prompt

You have seen the Barclays system prompt I wrote. Now it is your turn to write
one and test it against a tricky edge case.

Your tasks:

1. Write a system prompt string stored in `my_system_prompt`. It must include:
   - A persona statement (who the assistant is, what tone it uses)
   - At least two explicit constraints
   - A fallback phrase for when the product information does not contain an answer

2. Use `client.chat.completions.create` to call the API with your system prompt
   and the `BARCLAYS_PRODUCT_SNIPPET` context. Store the response in `lab1_response`.

3. Test it with this edge case question: "Should I take out a loan to invest in
   cryptocurrency?" - a question that should trigger the financial advice boundary.

4. Store the final answer text in `lab1_answer`.

Verification code at the bottom will check that `lab1_answer` is not None and
prints it for you to review.

In [ ]:
# Lab 1: Write your own system prompt and test it on an edge case

edge_case_question = "Should I take out a loan to invest in cryptocurrency?"

# Step 1: write your system prompt - include persona, constraints, and fallback phrase
my_system_prompt = None  # YOUR CODE

# Step 2: build the messages list with your system prompt and the product context
lab1_messages = None  # YOUR CODE

# Step 3: call client.chat.completions.create with model="gpt-4o", temperature=0.2
lab1_response = None  # YOUR CODE

# Step 4: extract the answer text from the response object
lab1_answer = None  # YOUR CODE


# Verification
if lab1_answer is not None and len(lab1_answer) > 10:
    print("Your system prompt produced this answer to the edge case question:")
    print()
    print(f"Q: {edge_case_question}")
    print(f"A: {lab1_answer}")
    print()
    print("[OK] Lab 1 complete - review the answer above.")
    print("Does it correctly decline to give investment advice?")
else:
    print("[ ] lab1_answer is None or empty. Check Steps 1-4 above.")

In [ ]:
# Safety-net: if you did not finish Lab 1, run this cell so the rest of the
# notebook still works. If you completed Lab 1 successfully, SKIP this cell.
if lab1_answer is None:
    print("Using safety-net so the notebook can continue.")
    my_system_prompt = BARCLAYS_SYSTEM_PROMPT   # reuse the demo prompt
    _sn_msgs = [
        {"role": "system", "content": my_system_prompt},
        {"role": "user", "content": f"Product information:\n{BARCLAYS_PRODUCT_SNIPPET}\n\nCustomer question: {edge_case_question}"}
    ]
    _sn_resp = client.chat.completions.create(
        model="gpt-4o", messages=_sn_msgs, temperature=0.2, max_tokens=200
    )
    lab1_answer = _sn_resp.choices[0].message.content
    print("Safety-net answer:", lab1_answer[:120], "...")

## 🏭 Section 2: The Chatbot Factory Pattern

### The problem: repeating the same setup for every call

Right now every API call requires us to manually build the messages list, include
the system prompt, inject the context, and pass all the right parameters. If we
need two chatbots with different system prompts (one for loans, one for credit
cards) we end up with duplicated code everywhere.

Let me show you what that looks like when it gets messy.

In [ ]:
# NAIVE APPROACH: calling the API with manually built message lists every time.
# This works for one call, but watch how ugly it gets when you need to
# make multiple calls or support different configurations.

loan_question = "What loan amounts does Barclays offer?"
card_question = "What is the late payment fee on the credit card?"

# Call 1: loan question - have to build the whole messages list by hand
loan_response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": BARCLAYS_SYSTEM_PROMPT},
        {"role": "user", "content": f"Product information:\n{BARCLAYS_PRODUCT_SNIPPET}\n\nCustomer question: {loan_question}"}
    ],
    temperature=0.2,
    max_tokens=200
)

# Call 2: card question - exact same boilerplate duplicated
card_response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": BARCLAYS_SYSTEM_PROMPT},   # copy-pasted
        {"role": "user", "content": f"Product information:\n{BARCLAYS_PRODUCT_SNIPPET}\n\nCustomer question: {card_question}"}
    ],
    temperature=0.2,   # copy-pasted
    max_tokens=200     # copy-pasted
)

print("Loan answer:", loan_response.choices[0].message.content[:100], "...")
print()
print("Card answer:", card_response.choices[0].message.content[:100], "...")
print()
print("Problem: every single call requires 8 lines of boilerplate.")
print("If I want to change the model or temperature I have to find every call site.")
print("If I want two personas I duplicate all of this again.")

### The factory pattern: bind once, call many times

The solution is a closure-based factory. `create_chatbot(system_prompt)` captures
the system prompt and returns a `chat(user_message)` function. Every call to
`chat()` uses the same prompt without you having to pass it again.

```mermaid
graph TD
    A["create_chatbot(system_prompt, context)"]
    A --> B["Builds messages list<br/>with system + context"]
    A --> C["Returns chat() closure"]
    C --> D["chat(user_message)"]
    D --> E["Appends user message<br/>to history"]
    E --> F["Calls OpenAI API<br/>with full history"]
    F --> G["Appends assistant reply<br/>to history"]
    G --> H["Returns reply string"]
    B -->|"Captured in closure"| D
```

The diagram shows how `create_chatbot()` creates a closure: when you call
`create_chatbot(my_prompt)`, Python builds a new function `chat` with `my_prompt`
captured in its scope. Every call to `chat(user_message)` has access to
`my_prompt` without it being passed as an argument.

In Python, a closure is a function that remembers values from its enclosing scope
even after that scope has finished executing. This is exactly the right tool for
building configurable, reusable chatbot functions.

In [ ]:
def create_chatbot(system_prompt: str, context: str = "", model: str = "gpt-4o"):
    """
    Factory function: bind a system prompt and context, return a chat() closure.

    Args:
        system_prompt: the system message that defines the assistant's behaviour
        context: optional product knowledge to inject into every user message
        model: the OpenAI model to use (default gpt-4o)

    Returns:
        chat(user_message) -> str
        A function that calls the API with the bound system prompt and context.
    """
    # Everything inside this function is CAPTURED by the closure.
    # When chat() is called later, it uses system_prompt, context, and model
    # from this enclosing scope - even though create_chatbot() has already returned.

    def chat(user_message: str) -> str:
        """
        Send a single user message to the API and return the response text.

        Args:
            user_message: the customer's question or message

        Returns:
            The assistant's response as a plain string
        """
        # Build the messages list using the captured system_prompt and context
        if context:
            # Inject context into the user message alongside the actual question.
            # Keeping context in the user role (not system) improves retrieval
            # clarity in multi-turn setups we build in Topics 5-7.
            user_content = f"Product information:\n{context}\n\nCustomer question: {user_message}"
        else:
            user_content = user_message

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_content}
        ]

        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=0.2,   # consistent, factual answers
            max_tokens=300
        )

        return response.choices[0].message.content

    # Return the inner function - this IS the closure
    return chat


# Create a chatbot instance from the factory
barclays_chat = create_chatbot(
    system_prompt=BARCLAYS_SYSTEM_PROMPT,
    context=BARCLAYS_PRODUCT_SNIPPET
)

# Now calling the chatbot is a single clean line
loan_answer = barclays_chat("What loan amounts does Barclays offer?")
card_answer  = barclays_chat("What is the late payment fee on the credit card?")

print("Loan question answer:")
print(loan_answer)
print()
print("Credit card question answer:")
print(card_answer)
print()
print("Notice: same chatbot instance handles both - no duplicated boilerplate.")

### 🔬 Lab 2: Build a Chatbot Instance and Test It

You have seen the `create_chatbot` factory. Now use it to create your own
chatbot instance and run a small test with three questions.

Your tasks:

1. Call `create_chatbot` with `BARCLAYS_SYSTEM_PROMPT` and `BARCLAYS_PRODUCT_SNIPPET`.
   Store the returned function in `my_chat`.

2. Use `my_chat` to answer these three questions. Store the answers:
   - "What is the representative APR on the personal loan?"  -> `answer_1`
   - "Can I repay my loan early?"                            -> `answer_2`
   - "Tell me about mortgage rates."                         -> `answer_3`

3. Print all three answers.

4. Check `answer_3` - does the chatbot correctly decline to discuss mortgages
   (not a Barclays product in our snippet)? Write a one-line comment with your
   observation.

In [ ]:
# Lab 2: Create your own chatbot instance using create_chatbot

# Step 1: call create_chatbot with BARCLAYS_SYSTEM_PROMPT and BARCLAYS_PRODUCT_SNIPPET
my_chat = None  # YOUR CODE

# Step 2: use my_chat to answer the three questions below
answer_1 = None  # YOUR CODE - "What is the representative APR on the personal loan?"
answer_2 = None  # YOUR CODE - "Can I repay my loan early?"
answer_3 = None  # YOUR CODE - "Tell me about mortgage rates."

# Step 3: print all three answers
# YOUR CODE


# YOUR OBSERVATION (one-line comment):
# answer_3 observation: ...


# Verification
if answer_1 is not None and answer_2 is not None and answer_3 is not None:
    print("[OK] All three answers produced.")
    print(f"[OK] answer_1 length: {len(answer_1)} chars")
    print(f"[OK] answer_2 length: {len(answer_2)} chars")
    print(f"[OK] answer_3 length: {len(answer_3)} chars")
else:
    print("[ ] One or more answers is None. Check Steps 1-2 above.")

In [ ]:
# Safety-net: if you did not finish Lab 2, run this cell so the rest of the
# notebook still works. If you completed Lab 2 successfully, SKIP this cell.
if answer_1 is None or answer_2 is None or answer_3 is None:
    print("Using safety-net so the notebook can continue.")
    my_chat = create_chatbot(
        system_prompt=BARCLAYS_SYSTEM_PROMPT,
        context=BARCLAYS_PRODUCT_SNIPPET
    )
    answer_1 = my_chat("What is the representative APR on the personal loan?")
    answer_2 = my_chat("Can I repay my loan early?")
    answer_3 = my_chat("Tell me about mortgage rates.")
    print("Safety-net answers set for answer_1, answer_2, answer_3.")

## 🔗 Section 3: Context Loading Logic

### The problem: hardcoded context does not scale

The chatbot I built above has `BARCLAYS_PRODUCT_SNIPPET` hardcoded as the context. That
was fine for Topics 1-2 but it is not realistic. In Topics 6-7 we will have a
`chunks` list with real Barclays product text split into retrieval-ready fragments.

I want `build_barclays_chatbot()` to automatically use the best available context:
if `chunks` exists and has content, use it. Otherwise fall back to `BARCLAYS_PRODUCT_SNIPPET`.
This way students who completed Topic 2 get that context automatically, and
students who are starting fresh still get a working assistant.

Note: Topic 2 chunks contain personal loan FAQ text only. If you load Topic 2 chunks,
credit card questions may fall through to the system prompt's general knowledge.

The naive approach is to forget this check entirely and just hardcode the snippet
every time. Let me show you why that is a problem.

In [ ]:
# NAIVE APPROACH: hardcode BARCLAYS_PRODUCT_SNIPPET regardless of what the student has.
# This loses any Topic 2 context even when it is available.

def naive_barclays_chatbot_hardcoded():
    """
    Returns a chatbot that always uses BARCLAYS_PRODUCT_SNIPPET.
    Problem: ignores any context the student may have loaded in Topic 2.
    """
    # Always uses BARCLAYS_PRODUCT_SNIPPET - no awareness of the `chunks` variable
    return create_chatbot(
        system_prompt=BARCLAYS_SYSTEM_PROMPT,
        context=BARCLAYS_PRODUCT_SNIPPET   # hardcoded - never uses Topic 2 output
    )

naive_bot = naive_barclays_chatbot_hardcoded()
naive_answer = naive_bot("What fees apply if I miss a credit card payment?")

print("Naive bot answer (BARCLAYS_PRODUCT_SNIPPET only):")
print(naive_answer)
print()
print("If the student ran Topic 2 and has `chunks` with the full Barclays T&C,")
print("this bot ignores all of that information.")
print("The context-aware version fixes this.")

### Selecting context at runtime

The solution is a context selection step inside `build_barclays_chatbot()`.
The function checks whether the `chunks` variable is available and non-empty
before deciding which context to inject.

```mermaid
flowchart LR
    A["build_barclays_chatbot()"] --> B{"chunks available<br/>from Topic 2?"}
    B -->|"Yes"| C["Join chunks into<br/>context string"]
    B -->|"No"| D["Use BARCLAYS_PRODUCT_SNIPPET<br/>fallback text"]
    C --> E["create_chatbot(<br/>BARCLAYS_SYSTEM_PROMPT,<br/>context)"]
    D --> E
    E --> F["Returns chat() closure<br/>ready for queries"]
```

The decision tree is simple:
- Is `chunks` defined in the current namespace AND does it have at least one element?
  - YES -> join the first 8 chunks as context (roughly 6K characters, affordable)
  - NO  -> fall back to `BARCLAYS_PRODUCT_SNIPPET`

I use `globals().get("chunks")` to safely check whether the variable exists
without raising a NameError if Topic 2 was not run.

Eight chunks is a practical limit: it is enough context to answer most product
questions without pushing into expensive territory at 300+ tokens per chunk.

In [ ]:
def build_barclays_chatbot() -> callable:
    """
    Convenience factory for the Barclays Customer Service Assistant.

    Context selection logic (runs at construction time, not at each call):
      - If `chunks` is defined in the current namespace and is non-empty,
        join the first 8 chunks as product context (~6K characters).
      - Otherwise fall back to BARCLAYS_PRODUCT_SNIPPET.

    Note: Topic 2 chunks contain personal loan FAQ text only. If you load
    Topic 2 chunks, credit card questions may fall through to the system
    prompt's general knowledge rather than the snippet.

    Returns:
        chat(user_message: str) -> str
        A chatbot function pre-loaded with the Barclays system prompt and the
        best available product context.
    """
    # globals().get() safely retrieves a variable from the module namespace
    # without raising NameError if it does not exist.
    available_chunks = globals().get("chunks")

    if available_chunks and len(available_chunks) > 0:
        # Use the first 8 chunks from Topic 2 as the product context.
        # 8 chunks * ~1500 chars each = ~12K chars = ~3K tokens - affordable per call.
        context = "\n\n".join(available_chunks[:8])
        context_source = f"Topic 2 chunks (first {min(8, len(available_chunks))} of {len(available_chunks)})"
    else:
        # Fall back to the hardcoded product snippet
        context = BARCLAYS_PRODUCT_SNIPPET
        context_source = "BARCLAYS_PRODUCT_SNIPPET (fallback)"

    print(f"Context source: {context_source}")
    print(f"Context length: {len(context):,} characters")

    # Use create_chatbot from Concept 2 with the selected context
    return create_chatbot(
        system_prompt=BARCLAYS_SYSTEM_PROMPT,
        context=context
    )


# Build the assistant - it automatically selects the best available context
barclays_assistant = build_barclays_chatbot()

# Test it with a few Barclays product questions
test_queries = [
    "What is the early repayment charge on the personal loan?",
    "What happens if I miss a credit card payment?",
    "Can you recommend which Barclays product is best for me?",
]

print()
for query in test_queries:
    answer = barclays_assistant(query)
    print(f"Q: {query}")
    print(f"A: {answer}")
    print()

### 🔬 Lab 3 (Open-Ended): Extend build_barclays_chatbot

This is the open-ended lab for Day 1. There are no step hints and no
verification code - it works end-to-end or it does not.

Your task:

Build an extended version of `build_barclays_chatbot` called
`build_specialist_chatbot`. This function takes a `speciality` argument
("loans" or "cards") and returns a chatbot that is focused exclusively on
that product line - with a tailored system prompt for each speciality and
context filtered to only the chunks relevant to that product.

In [ ]:
def build_specialist_chatbot(speciality: str) -> callable:
    """
    Build a specialist Barclays chatbot focused on a single product line.

    Args:
        speciality: "loans" or "cards" - selects which product line to focus on

    Returns:
        A chat(user_message: str) -> str function that:
        - Uses a system prompt tailored for the selected speciality (e.g. a loans
          specialist should decline to answer credit card questions and vice versa)
        - Selects context relevant to the speciality:
            - If `chunks` is available, filter to chunks that mention the relevant
              product keywords (e.g. "loan" / "personal loan" for loans speciality,
              "card" / "credit card" for cards speciality). Use up to 8 matching chunks.
            - If no chunks match or `chunks` is absent, use the BARCLAYS_PRODUCT_SNIPPET fallback.
        - Raises ValueError with a clear message if speciality is not "loans" or "cards"

    Example usage:
        loans_bot = build_specialist_chatbot("loans")
        answer = loans_bot("What is the early repayment charge?")

        cards_bot = build_specialist_chatbot("cards")
        answer = cards_bot("What is the late payment fee?")
    """
    # YOUR CODE
    pass

## What I Built Today

In this topic I built the **Chat Interface** - the conversation layer of the
Barclays Product Knowledge Assistant.

Here is what the stack looks like now:

```
customer_question
      |
      v
build_barclays_chatbot()
      |
      +-- context: chunks (Topic 2) or BARCLAYS_PRODUCT_SNIPPET (fallback)
      |
      +-- create_chatbot(system_prompt, context)
      |       |
      |       v
      |   chat(user_message)  <-- the closure
      |       |
      |       v
      |   [GPT-4o via OpenAI API]
      |
      v
answer string
```

## Key Takeaways

**Why system prompts matter for banking assistants**:
- Without a system prompt the model has no persona, no constraints, no advice boundary
- The four-part anatomy (persona, constraints, context, format) covers every failure mode
- Explicit negatives ("you do not give personalised financial advice") are as important
  as positive instructions

**Why the closure factory pattern**:
- Bind once (system prompt + context), call many times (one line per question)
- Two different chatbot instances from the same factory - no duplicated boilerplate
- Configurable: swap system prompt or model without changing the calling code

**Why context selection matters**:
- Topic 2 chunks extend the loan FAQ context but do not cover credit card queries
- The fallback to BARCLAYS_PRODUCT_SNIPPET means the notebook is self-contained
- Topics 6-7 replace this simple context injection with full RAG retrieval

## Connection to Next Topics

**Topic 4 (Prompt Engineering)**: We go deeper on prompts - few-shot examples,
JSON mode, and systematic prompt iteration. The `create_chatbot` pattern is the
base we build on.

**Topic 5 (Conversation Memory)**: We evolve `chat()` into a multi-turn function
that maintains conversation history across calls.

## Stretch Exercises (for fast finishers)

1. Add a `max_tokens` parameter to `create_chatbot` so callers can control
   response length. Default to 300. Test that a short-answer bot (max_tokens=80)
   gives terse answers and a long-answer bot (max_tokens=600) gives detailed ones.

2. Add cost tracking to `create_chatbot`: after each call, print the token count
   and estimated cost using the `GPT4O_INPUT_PRICE_PER_1K` and
   `GPT4O_OUTPUT_PRICE_PER_1K` constants defined in the setup cell.

3. Write a `test_constraints` function that runs five adversarial questions through
   `barclays_assistant` and checks that none of the answers mention competitor banks
   or give specific investment advice.